In [ ]:
import pandas as pd
from typing import Any
import requests
import pandas as pd
import matplotlib.pyplot as plt
from time import sleep
import os
from dotenv import load_dotenv
import psycopg
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor 

In [116]:
import re

def normalizar_hora(valor):
    # Comprobamos si es nulo antes de convertir a string
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip()
    if valor in ("<NA>", "nan", "NaN", "None", ""):
        return np.nan
    if re.fullmatch(r"\d{1,2}", valor):
        return f"{int(valor):02d}:00"
    return valor

def convertir_datos_aemet(df):
    df = df.copy()

    # Fecha a datetime
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")

    # Categóricas / texto
    for col in ["indicativo", "provincia", "nombre"]:
        if col in df.columns:
            df[col] = df[col].astype("category")

    columnas_float = [
        "tmed", "prec", "tmin", "tmax", "hrMedia", "pintMax",
        "velmedia", "racha", "presMax", "presMin", "sol"
    ]

    columnas_hora = [
        "horatmin", "horatmax", "horaHrMax", "horaHrMin",
        "horaracha", "horaPresMax", "horaPresMin", "horaPIntMax"
    ]

    columnas_a_revisar = [c for c in columnas_float + columnas_hora if c in df.columns]

    # Detectar Acum / Varias / Ip
    mask_acum = pd.Series(False, index=df.index)
    mask_varias = pd.Series(False, index=df.index)
    mask_ip = pd.Series(False, index=df.index)

    for col in columnas_a_revisar:
        serie = df[col].apply(lambda x: str(x).strip().lower() if pd.notna(x) else np.nan)
        mask_acum |= (serie == "acum")
        mask_varias |= (serie == "varias")
        mask_ip |= (serie == "ip")

    df["precAcum"] = mask_acum.fillna(False)
    df["variasHoras"] = mask_varias.fillna(False)
    df["precIp"] = mask_ip.fillna(False)

    # Limpieza numérica
    def limpiar_numero(valor):
        if pd.isna(valor):
            return np.nan
        valor = str(valor).strip()
        if valor.lower() in ("ip",):
            return "0.05"
        if valor.lower() in ("acum", "varias", "<na>", "nan", "none", ""):
            return np.nan
        return valor.replace(",", ".")

    for col in columnas_float:
        if col in df.columns:
            df[col] = df[col].apply(limpiar_numero)
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Numéricos -> float
    for col in ["altitud", "hrMax", "hrMin", "dir"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    # 5. Columnas de hora -> normalizadas a HH:MM
    for col in columnas_hora:
        if col in df.columns:
            df[col] = df[col].apply(normalizar_hora)

    return df

# df_datos_limpio = convertir_datos_aemet(df_datos)

pd.set_option('display.max_columns', None)
# df_datos_limpio.sample(50)

In [117]:
df=pd.read_pickle("ALL_10_YEARS")
df=convertir_datos_aemet(df)


In [118]:
df

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,dir,velmedia,racha,horaracha,presMax,horaPresMax,presMin,horaPresMin,sol,horaPIntMax,precAcum,variasHoras,precIp
0,2016-08-08,7250C,ABANILLA,MURCIA,174.0,25.0,0.0,17.3,04:22,32.7,Varias,46.0,80.0,00:00,32.0,13:30,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
1,2016-08-08,0255B,SANTA SUSANNA,BARCELONA,40.0,22.9,0.0,16.2,02:50,29.6,12:00,48.0,69.0,23:40,34.0,09:10,0.0,22.0,1.9,8.6,13:10,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
2,2016-08-08,5612B,LA RODA DE ANDALUCÍA,SEVILLA,410.0,26.4,0.0,19.5,05:40,33.2,14:00,38.0,69.0,05:40,24.0,Varias,0.0,18.0,5.3,14.2,16:20,973.5,08:00,970.3,18:00,NaN,NaN,False,True,False
3,2016-08-08,2885K,FRESNO DE SAYAGO,ZAMORA,804.0,25.8,0.0,15.7,05:13,36.0,15:53,26.0,60.0,Varias,13.0,13:20,0.0,5.0,4.4,11.7,08:40,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
4,2016-08-08,8492X,ATZENETA DEL MAESTRAT,CASTELLON,420.0,24.5,0.0,14.7,05:09,34.3,14:31,41.0,89.0,05:10,27.0,Varias,0.0,19.0,2.8,8.9,15:10,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3042388,2026-07-24,C665T,VALLESECO,LAS PALMAS,900.0,19.0,0.0,15.1,06:37,22.8,13:54,82.0,96.0,07:00,66.0,02:30,0.0,34.0,1.9,5.3,12:30,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
3042389,2026-07-24,4061X,QUINTANAR DE LA ORDEN,TOLEDO,691.0,28.4,0.0,21.7,23:59,35.0,14:30,24.0,39.0,07:20,14.0,Varias,0.0,20.0,3.1,13.1,13:20,937.5,07:00,934.6,Varias,NaN,NaN,False,True,False
3042390,2026-07-24,2096B,LICERAS,SORIA,1150.0,22.9,0.0,16.1,23:02,29.7,13:53,35.0,50.0,06:40,22.0,Varias,0.0,30.0,5.8,10.8,15:00,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
3042391,2026-07-24,2140A,ALDEANUEVA DE SERREZUELA,SEGOVIA,1135.0,21.7,0.0,16.0,23:56,27.4,14:40,39.0,65.0,05:50,23.0,Varias,0.0,27.0,5.0,13.6,15:10,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False


In [142]:
WINDOW_SIZE: int = 20

temp = df[df["indicativo"] == "7250C"].sort_values(by="fecha")[[
    "fecha",
    "tmed",
    "hrMedia"#,
    # "dia_sin",
    # "dia_cos"
    ]].reset_index(drop=True)
temp["dia_sin"] = np.sin(2*np.pi*temp["fecha"].dt.dayofyear/365)
temp["dia_cos"] = np.cos(2*np.pi*temp["fecha"].dt.dayofyear/365)

train = temp.iloc[10:-365]
test = temp.iloc[-365:]

rows = []
for i in range(train.shape[0]-WINDOW_SIZE):
    date=temp.iloc[i+WINDOW_SIZE]["fecha"]
    hrMedia=temp.iloc[i+WINDOW_SIZE]["hrMedia"]
    day_sin = temp.iloc[i+WINDOW_SIZE]["dia_sin"]
    day_cos = temp.iloc[i+WINDOW_SIZE]["dia_cos"]  
    days=temp["tmed"].loc[i:WINDOW_SIZE+i].to_numpy()

    final_row = np.hstack([
        date,
        day_sin,
        day_cos,
        hrMedia,
        days
    ])

    rows.append(final_row)


df_estacion=pd.DataFrame(rows)
df_estacion.iloc[:, 4] = pd.to_numeric(df_estacion.iloc[:, 4], errors="coerce")
df_estacion=df_estacion.dropna()
df_estacion_X : pd.DataFrame = df_estacion.iloc[:, 1:-1]
df_estacion_Y : pd.DataFrame = df_estacion.iloc[:, -1]
corte=365
df_estacion_X_train : pd.DataFrame =df_estacion_X.iloc[:-corte]
df_estacion_X_test : pd.DataFrame=df_estacion_X.iloc[-corte:]
df_estacion_Y_train : pd.DataFrame=df_estacion_Y.iloc[:-corte]
df_estacion_Y_test : pd.DataFrame=df_estacion_Y.iloc[-corte:]

In [138]:
df_estacion

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,2016-08-28,-0.845249,-0.534373,66.0,25.0,24.6,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9
1,2016-08-29,-0.854322,-0.519744,70.0,24.6,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8
2,2016-08-30,-0.863142,-0.504961,69.0,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4
3,2016-08-31,-0.871706,-0.490029,69.0,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0
4,2016-09-01,-0.880012,-0.474951,67.0,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0,24.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3238,2025-07-10,-0.145799,-0.989314,67.0,28.0,28.0,28.8,27.4,27.6,27.4,29.6,27.5,28.3,29.5,29.0,29.0,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6
3239,2025-07-11,-0.162807,-0.986658,68.0,28.0,28.8,27.4,27.6,27.4,29.6,27.5,28.3,29.5,29.0,29.0,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8
3240,2025-07-12,-0.179767,-0.983709,44.0,28.8,27.4,27.6,27.4,29.6,27.5,28.3,29.5,29.0,29.0,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8,29.1
3241,2025-07-13,-0.196673,-0.980469,57.0,27.4,27.6,27.4,29.6,27.5,28.3,29.5,29.0,29.0,28.6,28.3,28.2,27.8,28.6,28.3,29.4,28.2,27.6,27.8,29.1,27.4


In [139]:
df_estacion_X_train

,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
0,-0.845249,-0.534373,66.0,25.0,24.6,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1
1,-0.854322,-0.519744,70.0,24.6,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9
2,-0.863142,-0.504961,69.0,24.4,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8
3,-0.871706,-0.490029,69.0,25.3,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4
4,-0.880012,-0.474951,67.0,23.4,22.6,24.5,23.6,26.1,28.4,26.4,28.3,28.9,26.2,24.6,24.2,24.2,24.0,24.9,25.1,27.9,26.8,26.4,25.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2873,-0.162807,-0.986658,53.0,26.2,24.8,24.6,24.0,23.1,24.2,24.3,26.0,26.3,27.0,26.6,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7
2874,-0.179767,-0.983709,60.0,24.8,24.6,24.0,23.1,24.2,24.3,26.0,26.3,27.0,26.6,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0
2875,-0.196673,-0.980469,59.0,24.6,24.0,23.1,24.2,24.3,26.0,26.3,27.0,26.6,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0,27.0
2876,-0.213521,-0.976938,67.0,24.0,23.1,24.2,24.3,26.0,26.3,27.0,26.6,24.2,24.5,25.9,26.8,24.7,26.6,26.6,26.0,25.7,29.0,27.0,29.2


In [107]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

model: xgb.XGBRegressor = xgb.XGBRegressor(
    random_state=42,
    n_estimators=800,
    learning_rate=0.008
    )

model.fit(df_estacion_X_train, df_estacion_Y_train)
yhat: np.ndarray = model.predict(df_estacion_X_test)
print("MAE", mean_absolute_error(df_estacion_Y_test, yhat))
print("MSE", mean_squared_error(df_estacion_Y_test, yhat))
print("RMSE", root_mean_squared_error(df_estacion_Y_test, yhat))
print("R2", r2_score(df_estacion_Y_test, yhat))

MAE 1.2925420181065388
MSE 2.8925145232165885
RMSE 1.7007394048520745
R2 0.9268467164719186


In [140]:
modelo = RandomForestRegressor(
    n_estimators=1000,
    max_depth=10,
    random_state=42,
    min_samples_leaf=10,
    # criterion="poisson",
    bootstrap=True,
    n_jobs=-1
)

modelo.fit(df_estacion_X_train, df_estacion_Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"m

In [143]:
pred = modelo.predict(df_estacion_X_test)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MAE :", mean_absolute_error(df_estacion_Y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(df_estacion_Y_test, pred)))
print("R²  :", r2_score(df_estacion_Y_test, pred))

MAE : 1.289828911955962
RMSE: 1.7079333857785721
R²  : 0.926226543447013


In [121]:
df.head(30)

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,hrMedia,hrMax,horaHrMax,hrMin,horaHrMin,pintMax,dir,velmedia,racha,horaracha,presMax,horaPresMax,presMin,horaPresMin,sol,horaPIntMax,precAcum,variasHoras,precIp
0,2016-08-08,7250C,ABANILLA,MURCIA,174.0,25.0,0.0,17.3,04:22,32.7,Varias,46.0,80.0,00:00,32.0,13:30,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
1,2016-08-08,0255B,SANTA SUSANNA,BARCELONA,40.0,22.9,0.0,16.2,02:50,29.6,12:00,48.0,69.0,23:40,34.0,09:10,0.0,22.0,1.9,8.6,13:10,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
2,2016-08-08,5612B,LA RODA DE ANDALUCÍA,SEVILLA,410.0,26.4,0.0,19.5,05:40,33.2,14:00,38.0,69.0,05:40,24.0,Varias,0.0,18.0,5.3,14.2,16:20,973.5,08:00,970.3,18:00,NaN,NaN,False,True,False
3,2016-08-08,2885K,FRESNO DE SAYAGO,ZAMORA,804.0,25.8,0.0,15.7,05:13,36.0,15:53,26.0,60.0,Varias,13.0,13:20,0.0,5.0,4.4,11.7,08:40,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
4,2016-08-08,8492X,ATZENETA DEL MAESTRAT,CASTELLON,420.0,24.5,0.0,14.7,05:09,34.3,14:31,41.0,89.0,05:10,27.0,Varias,0.0,19.0,2.8,8.9,15:10,NaN,NaN,NaN,NaN,NaN,NaN,False,True,False
5,2016-08-08,2182C,PEDRAZA,SEGOVIA,1107.0,24.4,0.0,15.8,05:32,33.0,14:42,21.0,43.0,22:10,15.0,18:00,0.0,20.0,2.2,13.9,14:50,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
6,2016-08-08,8270X,BICORP,VALENCIA,305.0,25.4,0.0,17.2,05:32,33.7,14:13,34.0,82.0,02:40,21.0,15:20,0.0,8.0,1.9,7.2,16:00,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
7,2016-08-08,4520X,FREGENAL DE LA SIERRA,BADAJOZ,586.0,26.8,0.0,17.2,05:21,36.4,15:55,31.0,73.0,06:10,19.0,14:20,0.0,14.0,2.5,9.4,12:30,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
8,2016-08-08,2918Y,EL MAÍLLO,SALAMANCA,1027.0,25.1,0.0,17.0,04:48,33.2,14:21,18.0,54.0,04:00,12.0,17:20,0.0,4.0,3.6,9.7,12:50,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
9,2016-08-08,8293X,XÀTIVA,VALENCIA,88.0,27.7,NaN,20.5,03:30,34.9,15:40,34.0,82.0,01:10,14.0,17:30,NaN,31.0,3.1,8.6,14:50,1013.4,08:00,1009.0,18:00,13.2,NaN,False,False,False


In [122]:
def create_data_window(df:pd.DataFrame , date:str, indicativo:str, WINDOW :int = 10,) -> tuple[pd.DataFrame, float]:
    rows =[]
    df["dia_sin"] = np.sin(2*np.pi*df["fecha"].dt.dayofyear/365)
    df["dia_cos"] = np.cos(2*np.pi*df["fecha"].dt.dayofyear/365)
    df=df[df["indicativo"] == indicativo].sort_values(by="fecha")[["fecha","tmed","hrMedia","dia_sin","dia_cos"]].reset_index(drop=True)
    for i in range(train.shape[0]-WINDOW):
        dates=df.iloc[i+WINDOW]["fecha"]
        hrMedia=df.iloc[i+WINDOW]["hrMedia"]
        day_sin = df.iloc[i+WINDOW]["dia_sin"]
        day_cos = df.iloc[i+WINDOW]["dia_cos"]  
        days=df["tmed"].loc[i:WINDOW+i].to_numpy()

        final_row = np.hstack([
            dates,
            day_sin,
            day_cos,
            hrMedia,
            days
        ])
        rows.append(final_row)
        sdf=pd.DataFrame(rows)
        sdf = sdf[sdf.iloc[:, 0] == date]
        
    return (sdf.iloc[0, 1:-1], sdf.iloc[:, -1].iloc[0])

In [123]:
busca_b=create_data_window(df=df,date="2023-10-12",indicativo="7250C",WINDOW=20)
print(busca_b[0])
print(busca_b[1])

1    -0.981306
2     0.192452
3         73.0
4         23.2
5         21.2
6         21.5
7         21.2
8         23.0
9         23.0
10        22.6
11        22.9
12        23.4
13        24.2
14        23.0
15        23.0
16        23.8
17        22.8
18        22.6
19        21.6
20        22.4
21        21.5
22        21.9
23        22.1
Name: 2601, dtype: object
21.7


In [ ]:
# valor_a_predecir=create_data_window(
#         df=df,
#         date="2023-12-31",
#         indicativo="0034X",
#         WINDOW=20)
# prediccion_from_modelos = modelos["0034X"].predict((valor_a_predecir[0].to_numpy().reshape(1, -1)))
# print(prediccion_from_modelos[0])
# print(valor_a_predecir[1])

In [ ]:
df["indicativo"].nunique()

894

In [ ]:
#ENTRENADOR Y EVALUADOR DE MODELO RANDOM FOREST

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae=[]
rmse=[]
r2=[]

modelos = {}
estaciones_no_modelas=[]
cont=0
n_cont=0
for indicativo, df_estacion in df.groupby("indicativo"):

    
    # if cont>20:
    #     continue

    WINDOW_SIZE: int = 20

    df_estacion["dia_sin"] = np.sin(2*np.pi*df_estacion["fecha"].dt.dayofyear/365)
    df_estacion["dia_cos"] = np.cos(2*np.pi*df_estacion["fecha"].dt.dayofyear/365)

    temp = df_estacion.sort_values(by="fecha")[[
        "fecha",
        "tmed",
        "hrMedia",
        "dia_sin",
        "dia_cos"
        ]].reset_index(drop=True)


    train = temp.iloc[10:-365]
    test = temp.iloc[-365:]

    rows = []
    for i in range(train.shape[0]-WINDOW_SIZE):
        date=temp.iloc[i+WINDOW_SIZE]["fecha"]
        hrMedia=temp.iloc[i+WINDOW_SIZE]["hrMedia"]
        day_sin = temp.iloc[i+WINDOW_SIZE]["dia_sin"]
        day_cos = temp.iloc[i+WINDOW_SIZE]["dia_cos"]  
        days=temp["tmed"].loc[i:WINDOW_SIZE+i].to_numpy()

        final_row = np.hstack([
            date,
            day_sin,
            day_cos,
            hrMedia,
            days
        ])

        rows.append(final_row)


    df_estacion=pd.DataFrame(rows)
    df_estacion=df_estacion.dropna()
    if len(df_estacion) < 1500:
        estaciones_no_modelas.append(indicativo)
        cont=1+cont
        # n_cont=1+n_cont
        print(f"La estación {indicativo} no tiene suficientes datos. Total de estaciones no modeladas {len(estaciones_no_modelas)}")
        continue
 

    df_estacion_X : pd.DataFrame = df_estacion.iloc[:, 1:-1]
    df_estacion_Y : pd.DataFrame = df_estacion.iloc[:, -1]
    corte=365
    df_estacion_X_train : pd.DataFrame =df_estacion_X.iloc[:-corte]
    df_estacion_X_test : pd.DataFrame=df_estacion_X.iloc[-corte:]
    df_estacion_Y_train : pd.DataFrame=df_estacion_Y.iloc[:-corte]
    df_estacion_Y_test : pd.DataFrame=df_estacion_Y.iloc[-corte:]

   

    X = df_estacion_X_train
    X_test=df_estacion_X_test
    y = df_estacion_Y_train
    y_test= df_estacion_Y_test

    modelo = RandomForestRegressor(
        n_estimators=1000,
        max_depth=10,
        random_state=42,
        min_samples_leaf=10,
        # criterion="poisson",
        bootstrap=True,
        n_jobs=-1
    )

    modelo.fit(X, y)
    pred = modelo.predict(df_estacion_X_test)
    

    cont=1+cont
    print(f"Modelo de estación {indicativo} entrenado. Total: {cont} de {df['indicativo'].nunique()}. Total de estaciones no modeladas {len(estaciones_no_modelas)}")
    mae.append(mean_absolute_error(df_estacion_Y_test, pred))
    print(f"MAE_ABSOLUTO: {np.mean(mae)} MAE de {indicativo}: {mean_absolute_error(df_estacion_Y_test, pred)}")
    rmse.append(np.sqrt(mean_squared_error(df_estacion_Y_test, pred)))
    print(f"RMSE_ABSOLUTO: {np.mean(rmse)} RMSE de {indicativo}: {np.sqrt(mean_squared_error(df_estacion_Y_test, pred))}")
    r2.append(r2_score(df_estacion_Y_test, pred))
    print(f"R²_ABSOLUTO: {np.mean(r2)} R² de {indicativo}: {r2_score(df_estacion_Y_test, pred)}")

    modelos[indicativo] = modelo

La estación 0002I no tiene suficientes datos. Total de estaciones no modeladas 1
Modelo de estación 0009X entrenado. Total: 2 de 894. Total de estaciones no modeladas 1
MAE_ABSOLUTO: 1.3586111602998086 MAE de 0009X: 1.3586111602998086
RMSE_ABSOLUTO: 1.784337598468787 RMSE de 0009X: 1.784337598468787
R²_ABSOLUTO: 0.9223742986853958 R² de 0009X: 0.9223742986853958
Modelo de estación 0016A entrenado. Total: 3 de 894. Total de estaciones no modeladas 1
MAE_ABSOLUTO: 1.2768500435057186 MAE de 0016A: 1.1950889267116287
RMSE_ABSOLUTO: 1.6694378667971232 RMSE de 0016A: 1.5545381351254597
R²_ABSOLUTO: 0.9293526162064248 R² de 0016A: 0.9363309337274539
Modelo de estación 0016B entrenado. Total: 4 de 894. Total de estaciones no modeladas 1
MAE_ABSOLUTO: 1.2329904896428465 MAE de 0016B: 1.145271381917102
RMSE_ABSOLUTO: 1.6105558814220815 RMSE de 0016B: 1.492791910671998
R²_ABSOLUTO: 0.9283268175462908 R² de 0016B: 0.9262752202260227
Modelo de estación 0034X entrenado. Total: 5 de 894. Total de est

[1.358611160299808,
 1.1950889267116287,
 1.145271381917102,
 1.1750682461676458,
 1.1613729309105874,
 1.3467888184565795,
 1.1927049847734514,
 1.062500537911691,
 1.0074553297132662,
 1.387217797542875,
 1.3269216727923379,
 1.3275378224320853,
 1.2984933898642004,
 1.270585412845271,
 1.291411408731432,
 1.339077848764794,
 1.332079772900544,
 1.4187072710231954,
 1.3901889864316936,
 1.253352322327407,
 1.0742503110287882,
 0.9387608127047536,
 1.150162197783727,
 0.992429345765052,
 1.0458510118247057,
 1.367499638257592,
 1.0464005729331483,
 1.169613490931044,
 1.3548843837226618,
 1.3256334126542635,
 1.3847831834266837,
 1.2761184342093512,
 1.4875388994193097,
 1.254167267846883,
 1.4342747252119996,
 1.3882141981901865,
 1.2492381289197223,
 1.3121857564050776,
 1.2166826542646039,
 1.210498787422642,
 1.3984217412752864,
 1.2530577021173521,
 1.3559534407920926,
 1.2161125363244238,
 1.4418987094900542,
 1.465375324208405,
 1.3538264419651185,
 1.2666020317647322,
 1.60458

In [ ]:
modelos

In [162]:
from pathlib import Path
from joblib import dump

# Carpeta donde se guardarán los modelos
carpeta_modelos = Path("modelos_GB")
carpeta_modelos.mkdir(exist_ok=True)

indicativos = []

for indicativo, modelo in modelos.items():

    # Guardar el modelo con el nombre del indicativo
    dump(modelo, carpeta_modelos / f"{indicativo}.joblib")

    indicativos.append(indicativo)

# Guardar la lista de indicativos
with open(carpeta_modelos / "indicativos.txt", "w", encoding="utf-8") as f:
    for indicativo in indicativos:
        f.write(f"{indicativo}\n")


In [ ]:
from pathlib import Path
from joblib import dump

# Carpeta donde se guardarán los modelos

carpeta_modelos = Path("modelos_RF")
carpeta_modelos.mkdir(parents=True, exist_ok=True)
indicativos = []
metricas_modelos = {}

# Comprobar que hay una métrica por cada modelo

if not (
    len(modelos) == len(mae) == len(rmse) == len(r2)
):
    raise ValueError(

        "El número de modelos no coincide con el número de métricas. "

        f"Modelos: {len(modelos)}, MAE: {len(mae)}, "

        f"RMSE: {len(rmse)}, R²: {len(r2)}"

    )

for posicion, (indicativo, modelo) in enumerate(modelos.items()):

    indicativo = str(indicativo)

    # Guardar el modelo con el nombre del indicativo

    dump(

        modelo,

        carpeta_modelos / f"{indicativo}.joblib"

    )

    indicativos.append(indicativo)

    # Guardar las métricas correspondientes a ese modelo

    metricas_modelos[indicativo] = {

        "MAE": float(mae[posicion]),

        "RMSE": float(rmse[posicion]),

        "R2": float(r2[posicion])

    }

# Guardar la lista de indicativos

with open(
    carpeta_modelos / "indicativos.txt","w",encoding="utf-8") as archivo:
    for indicativo in indicativos:
        archivo.write(f"{indicativo}\n")

# Guardar el diccionario de métricas
dump(metricas_modelos,carpeta_modelos / "metricas_modelos.joblib")

print(f"Modelos guardados: {len(indicativos)}")
print(f"Métricas guardadas: {len(metricas_modelos)}")

Modelos guardados: 820
Métricas guardadas: 820


In [159]:
from joblib import load

metricas_modelos = load("modelos_RF/metricas_modelos.joblib")

mae_medio = np.mean([metricas["MAE"] for metricas in metricas_modelos.values()])
rmse_medio = np.mean([metricas["RMSE"] for metricas in metricas_modelos.values()])
r2_medio = np.mean([metricas["R2"] for metricas in metricas_modelos.values()])

print(f"MAE medio: {mae_medio}")
print(f"RMSE medio: {rmse_medio}")
print(f"R² medio: {r2_medio}")

MAE medio: 1.3540692692741436
RMSE medio: 1.7277470935507964
R² medio: 0.9111918207419355


In [152]:
from pathlib import Path
from joblib import load

def cargar_modelo(indicativo: str):
    ruta = Path("modelos_RF") / f"{indicativo}.joblib"
    if not ruta.exists():
        raise FileNotFoundError(f"No existe un modelo para el indicativo {indicativo}")

    return load(ruta)

In [153]:
modelo_cargado = cargar_modelo("0034X")

In [154]:
valor_a_predecir=create_data_window(
        df=df,
        date="2023-12-31",
        indicativo="0034X",
        WINDOW=20)
prediccion_from_modelos = modelos["0034X"].predict((valor_a_predecir[0].to_numpy().reshape(1, -1)))
print(prediccion_from_modelos[0])
print(valor_a_predecir[1])

9.258559740898306
8.2


In [155]:
valor_a_predecir=create_data_window(
        df=df,
        date="2023-12-31",
        indicativo="0034X",
        WINDOW=20)
prediccion_from_modelos = modelo_cargado.predict((valor_a_predecir[0].to_numpy().reshape(1, -1)))
print(prediccion_from_modelos[0])
print(valor_a_predecir[1])

9.258559740898306
8.2
